# RQ analysis
**Q1**: Spearman rank correlation of latency/energy across the 3 input domains, per framework.

**Q2**: Spearman rank correlation of latency/energy across the 3 frameworks, per input domain.

**Q3**: Jaccard similarity of Pareto-optimal arch sets across (task, framework); shared subset.

Loads via `api.load(non_isomorphic=NON_ISOMORPHIC)` (long-form, columns `arch_idx, device, framework, task, lat_ms, lat_ms_var, energy_mj, accuracy`); `NON_ISOMORPHIC=True` restricts to the ~6466 non-isomorphic NB201 representatives. Selects one `DEVICE`. On load `framework→runtime`, `lat_ms→lat_ms_median` so the rest of the notebook is unchanged.

In [ ]:
from pathlib import Path
import sys, itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from utils.task_specs import TASKS
import api

DEVICE = 'jetson'
NON_ISOMORPHIC = True  # restrict to the ~6466 non-isomorphic NB201 representatives

df = api.load(non_isomorphic=NON_ISOMORPHIC)
# parquet is long-format & already filtered to ok rows. Rename to the column
# names the rest of the notebook expects (runtime / lat_ms_median).
df = df.rename(columns={'framework': 'runtime', 'lat_ms': 'lat_ms_median'})
df = df[df.device == DEVICE].copy()
TASK_LIST = [t for t in TASKS if t in df.task.unique()]
RUNTIMES = sorted(df.runtime.unique())
HAS_ENERGY = 'energy_mj' in df.columns and df.energy_mj.notna().any()

# pretty labels
TASK_LABEL = {'cifar100': 'CIFAR100', 'ninapro': 'NinaPro', 'darcy': 'DarcyFlow'}
RUNTIME_LABEL = {'litert': 'LiteRT', 'onnx': 'ONNX', 'torchmobile': 'Torchmobile'}
# darcy is regression: parquet accuracy = raw relative L2 error (lower better)
SCORE_LABEL = {'cifar100': 'accuracy', 'ninapro': 'accuracy', 'darcy': 'rel L2'}
tlab = lambda t: TASK_LABEL.get(t, t)
rlab = lambda r: RUNTIME_LABEL.get(r, r)

print(f'device: {DEVICE}  non_iso: {NON_ISOMORPHIC}  rows: {len(df)}  archs: {df.arch_idx.nunique()}  tasks: {TASK_LIST}  runtimes: {RUNTIMES}  energy: {HAS_ENERGY}')

## Helpers

In [ ]:
# pretty labels (kept here so plotting cells work without re-running cell 1)
TASK_LABEL = {'cifar100': 'CIFAR100', 'ninapro': 'NinaPro', 'darcy': 'DarcyFlow'}
RUNTIME_LABEL = {'litert': 'LiteRT', 'onnx': 'ONNX', 'torchmobile': 'Torchmobile'}
SCORE_LABEL = {'cifar100': 'accuracy', 'ninapro': 'accuracy', 'darcy': 'rel L2'}
# True = higher score is better, False = lower score is better.
# darcy: parquet accuracy column = raw relative L2 error -> lower better.
SCORE_HIGHER_BETTER = {'cifar100': True, 'ninapro': True, 'darcy': False}
tlab = lambda t: TASK_LABEL.get(t, t)
rlab = lambda r: RUNTIME_LABEL.get(r, r)

def pivot_metric(df, metric, index='arch_idx', col_a='task', col_b='runtime'):
    """Return wide df: rows = arch, cols = (col_a, col_b), vals = metric."""
    return df.pivot_table(index=index, columns=[col_a, col_b], values=metric)

def spearman_matrix(wide):
    """Spearman correlation matrix over columns of `wide`. NaN-safe via pairwise."""
    cols = list(wide.columns)
    n = len(cols)
    M = np.full((n, n), np.nan)
    for i, j in itertools.product(range(n), range(n)):
        a, b = wide[cols[i]], wide[cols[j]]
        mask = a.notna() & b.notna()
        if mask.sum() < 3: continue
        rho, _ = spearmanr(a[mask], b[mask])
        M[i, j] = rho
    return pd.DataFrame(M, index=cols, columns=cols)

def heatmap(M, title, ax=None, vmin=-1, vmax=1):
    if ax is None: fig, ax = plt.subplots(figsize=(4, 3.5))
    im = ax.imshow(M.values, vmin=vmin, vmax=vmax, cmap='RdBu_r')
    ax.set_xticks(range(len(M.columns))); ax.set_xticklabels(M.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(M.index))); ax.set_yticklabels(M.index)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8,
                        color='white' if abs(v) > 0.5 else 'black')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

## Q1: rank correlation across tasks, per framework
For each framework, take latency (and energy) per arch across the 3 tasks, then Spearman across task pairs.

In [ ]:
def q1(metric, label):
    fig, axes = plt.subplots(1, len(RUNTIMES), figsize=(4.2*len(RUNTIMES), 3.5))
    if len(RUNTIMES) == 1: axes = [axes]
    results = {}
    for ax, rt in zip(axes, RUNTIMES):
        sub = df[df.runtime == rt]
        wide = sub.pivot_table(index='arch_idx', columns='task', values=metric)
        wide = wide.rename(columns=TASK_LABEL)
        M = spearman_matrix(wide)
        results[rt] = M
        heatmap(M, rlab(rt), ax=ax)
    plt.suptitle(f'{label} | {DEVICE}', y=1.01)
    plt.tight_layout(); plt.show()
    return results

q1_lat = q1('lat_ms_median', 'latency rank')

In [ ]:
if HAS_ENERGY:
    q1_eng = q1('energy_mj', 'energy rank')
else:
    print('energy column missing - skip')

## Q2: rank correlation across frameworks, per task

In [ ]:
def q2(metric, label):
    fig, axes = plt.subplots(1, len(TASK_LIST), figsize=(4.2*len(TASK_LIST), 3.5))
    if len(TASK_LIST) == 1: axes = [axes]
    results = {}
    for ax, t in zip(axes, TASK_LIST):
        sub = df[df.task == t]
        wide = sub.pivot_table(index='arch_idx', columns='runtime', values=metric)
        wide = wide.rename(columns=RUNTIME_LABEL)
        M = spearman_matrix(wide)
        results[t] = M
        heatmap(M, tlab(t), ax=ax)
    plt.suptitle(f'{label} | {DEVICE}', y=1.01)
    plt.tight_layout(); plt.show()
    return results

q2_lat = q2('lat_ms_median', 'latency rank')

In [ ]:
if HAS_ENERGY:
    q2_eng = q2('energy_mj', 'energy rank')
else:
    print('energy column missing - skip')

## Q3: Pareto-optimal sets across (task, framework)
Pareto: minimize latency (or energy), maximize accuracy from the long-form `accuracy` column.

In [ ]:
def acc_by_task_from_df(df, tasks):
    # parquet stores accuracy long-form: one `accuracy` value per (arch_idx, task).
    acc_by_task = {}
    for t in tasks:
        sub = df[df.task == t][["arch_idx", "accuracy"]].dropna()
        if sub.empty:
            acc_by_task[t] = None
            continue
        acc_by_task[t] = dict(
            sub.drop_duplicates("arch_idx").set_index("arch_idx")["accuracy"]
        )
    return acc_by_task

ACC_BY_TASK = acc_by_task_from_df(df, TASK_LIST)
print({t: f'n={len(v)}' if v else 'missing' for t, v in ACC_BY_TASK.items()})

In [ ]:
def pareto_front(points):
    """points: array (n, d) to minimize on all dims. Returns boolean mask of non-dominated."""
    P = np.asarray(points)
    n = P.shape[0]
    keep = np.ones(n, dtype=bool)
    for i in range(n):
        if not keep[i]: continue
        dom = np.all(P <= P[i], axis=1) & np.any(P < P[i], axis=1)
        if dom.any(): keep[i] = False
    return keep

def pareto_front_2d(cost, score, higher_better=True):
    """Indices on Pareto front: minimize cost, and maximize (or minimize) score."""
    # Lexsort sorts by the last key first (cost), then the first key (-score or score for tiebreakers)
    order = np.lexsort((-score if higher_better else score, cost))
    keep = []
    if higher_better:
        best = -np.inf
        for i in order:
            if score[i] > best: keep.append(i); best = score[i]
    else:
        best = np.inf
        for i in order:
            if score[i] < best: keep.append(i); best = score[i]
    return keep

def pareto_set(task, runtime, metric='lat_ms_median'):
    """Pareto front for (task, runtime) over (cost=metric, score).
    Score direction per task via SCORE_HIGHER_BETTER (darcy = lower better).
    """
    sub = df[(df.task == task) & (df.runtime == runtime)][['arch_idx', metric]].dropna()
    acc = ACC_BY_TASK.get(task)
    if acc is None:
        if sub.empty: return set()
        return {int(sub.loc[sub[metric].idxmin(), 'arch_idx'])}
    sub = sub.assign(acc=sub.arch_idx.map(acc)).dropna()
    if sub.empty: return set()
    sign = -1.0 if SCORE_HIGHER_BETTER.get(task, True) else 1.0  # convert score to minimize
    pts = np.column_stack([sub[metric].values, sign * sub['acc'].values])
    mask = pareto_front(pts)
    return set(sub.arch_idx.values[mask].astype(int))

In [ ]:
def jaccard(a, b):
    if not a and not b: return np.nan
    return len(a & b) / len(a | b)

def q3(metric, label):
    keys = [(t, rt) for t in TASK_LIST for rt in RUNTIMES]
    sets = {k: pareto_set(*k, metric=metric) for k in keys}
    labels = [f'{tlab(t)}/{rlab(rt)}' for t,rt in keys]
    n = len(keys)
    J = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            J[i,j] = jaccard(sets[keys[i]], sets[keys[j]])
    JM = pd.DataFrame(J, index=labels, columns=labels)
    fig, ax = plt.subplots(figsize=(0.6*n+2, 0.6*n+1.5))
    heatmap(JM, f'Jaccard | Pareto({label}, score)', ax=ax, vmin=0, vmax=1)
    plt.suptitle(f'Jaccard similarity | {DEVICE}', y=1.01)
    plt.tight_layout(); plt.show()
    common = set.intersection(*sets.values()) if all(sets.values()) else set()
    for k,v in sets.items(): print(f'  {tlab(k[0])}/{rlab(k[1])}: |P|={len(v)}')
    print(f'common across all (task, framework): {sorted(common)}')
    return sets, JM, common

q3_lat = q3('lat_ms_median', 'latency')

In [ ]:
if HAS_ENERGY:
    q3_eng = q3('energy_mj', 'energy')
else:
    print('energy column missing - skip')

## Scatter: accuracy vs latency / energy, per task
One panel per task; one color per framework. Pareto front (minimize cost, maximize accuracy) drawn as a step line.

In [ ]:
def scatter_acc_vs(metric, label):
    colors = plt.cm.tab10(np.linspace(0, 1, len(RUNTIMES)))
    rt_color = dict(zip(RUNTIMES, colors))

    fig, axes = plt.subplots(len(RUNTIMES), len(TASK_LIST),
                             figsize=(4.2 * len(TASK_LIST), 3.5 * len(RUNTIMES)),
                             squeeze=False)

    for ri, rt in enumerate(RUNTIMES):
        for ci, t in enumerate(TASK_LIST):
            ax = axes[ri][ci]
            acc_map = ACC_BY_TASK.get(t)
            hb = SCORE_HIGHER_BETTER.get(t, True)
            if not acc_map:
                ax.set_title(f'{tlab(t)}: no score'); continue

            sub = df[(df.task == t) & (df.runtime == rt)][['arch_idx', metric]].dropna()
            sub = sub.assign(score=sub.arch_idx.map(acc_map)).dropna()
            if sub.empty: continue

            ax.scatter(sub[metric], sub.score, s=6, alpha=0.35,
                       color=rt_color[rt], edgecolors='none')

            idx = pareto_front_2d(sub[metric].values, sub.score.values, higher_better=hb)
            pc = sub[metric].values[idx]
            ps = sub.score.values[idx]
            ax.scatter(pc, ps, s=18, color='black', zorder=5)

            if not hb: ax.invert_yaxis()
            ax.set_xlabel(label)
            ax.set_ylabel(SCORE_LABEL.get(t, 'score'))
            ax.set_title(f'{rlab(rt)} - {tlab(t)}')

    plt.suptitle(f'Score vs {label} | {DEVICE}', y=1.01)
    plt.tight_layout()
    plt.show()

scatter_acc_vs('lat_ms_median', 'latency (ms)')
if HAS_ENERGY:
    scatter_acc_vs('energy_mj', 'energy (mJ)')
else:
    print('energy column missing - skip')

In [ ]:
rows = []
for metric, label, result in [
    ('lat_ms_median', 'latency', q3_lat),
    *([('energy_mj', 'energy', q3_eng)] if HAS_ENERGY else []),
]:
    for (t, rt), s in result[0].items():
        rows.append({'metric': label, 'task': tlab(t), 'runtime': rlab(rt), '|P|': len(s)})

pd.DataFrame(rows).pivot_table(index=['task', 'runtime'], columns='metric', values='|P|', aggfunc='first')

## Op-frequency bias on the Pareto front
For each (task, framework), take Pareto-optimal archs (minimize cost, optimize score per task).
Parse NB201 arch_str into 6 ops, count freq of {none, skip_connect, nor_conv_1x1, nor_conv_3x3, avg_pool_3x3}.
Stacked bar per task compares Pareto-front op-mix vs full search-space op-mix across frameworks.

In [ ]:
import pickle, warnings, re
HW_PKL = ROOT / 'data' / 'hw-nas-bench' / 'HW-NAS-Bench-v1_0.pickle'
with open(HW_PKL, 'rb') as f, warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    _hw = pickle.load(f)
_configs = _hw['nasbench201']['cifar10']['config']
_iter = _configs.items() if isinstance(_configs, dict) else enumerate(_configs)
ARCH_STR = {int(i): cfg['arch_str'] for i, cfg in _iter}

OPS = ['none', 'skip_connect', 'nor_conv_1x1', 'nor_conv_3x3', 'avg_pool_3x3']
OP_LABEL = {'none': 'zeroize', 'skip_connect': 'skip', 'nor_conv_1x1': '1x1 conv',
            'nor_conv_3x3': '3x3 conv', 'avg_pool_3x3': 'avg pool'}
OP_COLOR = dict(zip(OPS, ['#bbbbbb', '#1f77b4', '#2ca02c', '#d62728', '#ff7f0e']))

# adjacent ops share a '|' separator (|a~0|b~1|), so anchoring with '|' on both sides
# consumes the boundary and skips every other op. Match by '~' instead.
_op_re = re.compile(r'([a-z_0-9]+?)~\d+')
def parse_ops(arch_str):
    return _op_re.findall(arch_str)

# sanity check: each arch has 6 ops
_lens = {len(parse_ops(s)) for s in ARCH_STR.values()}
_bad = {o for s in ARCH_STR.values() for o in parse_ops(s)} - set(OPS)
print(f'arch_str loaded: {len(ARCH_STR)}  op-count set: {_lens}  unknown ops: {_bad}')

def op_freq(arch_indices):
    """Return dict op -> fraction across all 6 edges of given archs. Normalized to sum 1."""
    c = {o: 0 for o in OPS}
    total = 0
    for ai in arch_indices:
        s = ARCH_STR.get(int(ai))
        if s is None: continue
        for o in parse_ops(s):
            if o in c: c[o] += 1; total += 1
    if total == 0: return c
    return {o: c[o] / total for o in OPS}

In [ ]:
def op_freq_bars(metric, label):
    """One panel per task. Bars: 'all' + one per framework's Pareto front. Stacked by op."""
    all_archs = sorted(df.arch_idx.unique())
    base = op_freq(all_archs)

    fig, axes = plt.subplots(1, len(TASK_LIST), figsize=(4.6*len(TASK_LIST), 4), squeeze=False)
    axes = axes[0]
    rows = []
    for ax, t in zip(axes, TASK_LIST):
        cols = ['all'] + [rlab(rt) for rt in RUNTIMES]
        freqs = [base]
        sizes = [len(all_archs)]
        for rt in RUNTIMES:
            P = pareto_set(t, rt, metric=metric)
            freqs.append(op_freq(P))
            sizes.append(len(P))
            for o in OPS:
                rows.append({'task': t, 'runtime': rt, 'op': o, 'freq': freqs[-1][o], 'n': len(P)})
        for o in OPS:
            rows.append({'task': t, 'runtime': 'all', 'op': o, 'freq': base[o], 'n': len(all_archs)})

        x = np.arange(len(cols))
        bottom = np.zeros(len(cols))
        for o in OPS:
            vals = np.array([f[o] for f in freqs])
            ax.bar(x, vals, bottom=bottom, color=OP_COLOR[o], label=OP_LABEL[o],
                   edgecolor='white', linewidth=0.5)
            for xi, (v, b) in enumerate(zip(vals, bottom)):
                if v > 0.05:
                    ax.text(xi, b + v/2, f'{v*100:.0f}', ha='center', va='center',
                            fontsize=7, color='white')
            bottom += vals
        ax.set_xticks(x)
        ax.set_xticklabels(cols, fontsize=8)
        ax.set_ylim(0, 1)
        ax.set_ylabel('op frequency (fraction of 6 edges)')
        ax.set_title(tlab(t))
    axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8, title='operation')
    plt.suptitle(f'{label} | {DEVICE}', y=1.01)
    plt.tight_layout(); plt.show()
    return pd.DataFrame(rows)

opfreq_lat = op_freq_bars('lat_ms_median', 'Pareto(latency, score)')
if HAS_ENERGY:
    opfreq_eng = op_freq_bars('energy_mj', 'Pareto(energy, score)')
else:
    print('energy column missing - skip')

In [ ]:
metrics = ['lat_ms_median', 'energy_mj'] if HAS_ENERGY else ['lat_ms_median']
agg = (
    df.groupby(['task', 'runtime'])[metrics]
    .agg(['mean', 'median', 'std'])
    .round(3)
)
agg.columns = ['_'.join(c) for c in agg.columns]
agg.columns = [c.replace('lat_ms_median_', 'lat_ms_') for c in agg.columns]
agg = agg.rename(index={**TASK_LABEL, **RUNTIME_LABEL})
agg